In [80]:
import re 
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from bs4 import BeautifulSoup

In [81]:
with open("apple_10k_2023.htm", 'r', encoding='utf-8' ,errors='ignore') as f:
    html  = f.read()

In [82]:
soup = BeautifulSoup(html,'html.parser')
for tag in soup(['script','style','meta','link']):
    tag.decompose()

In [83]:
print("HTML loaded and cleaned")
print(f"Total text length: {len(soup.get_text()):,} characters")

HTML loaded and cleaned
Total text length: 213,113 characters


#  TABLE TO NATURAL LANGUAGE ENGINE

In [84]:
class TableToNaturalLanguage:
    def __init__(self,company):
        """
        Initializes the converter with specific document metadata.
        """
        self.company = company
        
    def is_financial_table(self,table_soup):
        """
        Analyzes a table to determine if it contains financial data 
        based on keyword presence and numerical patterns.
        """
        financial_keywords = ["net", "revenue", "income", "sales", "earnings", 
                          "profit", "loss", "assets", "cash", "cost"]
        numeric_pattern =re.compile(r'\$?[\d,]+\.?\d*')

        rows = table_soup.find_all("tr")

        for row in rows:
            cells = row.find_all("td")
            if len(cells) < 2:
                continue

            first_cell_text = cells[0].get_text(strip=True).lower()
            has_keyword = any(kw in first_cell_text for kw in financial_keywords)

            has_numeric_value = False
            for cell in cells[1:]:
                cell_text = cell.get_text(strip=True)
                # We look for a cell that isn't empty and matches our financial number pattern
                if cell_text and numeric_pattern.search(cell_text):
                    has_numeric_value = True
                    break

            # If a single row contains both a keyword and a financial number, it's a winner
            if has_keyword and has_numeric_value:
                return True

        return False

    
    def convert(self, raw_html):
        soup = BeautifulSoup(raw_html, "html.parser")
        if not self.is_financial_table(soup):
            return []

        rows = soup.find_all("tr")
        col_map = {} # Temporary to find years and their order
        year_pattern = re.compile(r'20\d{2}')
        results = []

        # 1. Header Scan (Still needed to find WHICH years are in the table)
        for row in rows:
            cells = row.find_all(["td", "th"])
            for i, cell in enumerate(cells):
                text = cell.get_text(strip=True)
                if year_pattern.fullmatch(text):
                    col_map[i] = text
            if col_map: break

        if not col_map: return []
        
        # The One-Liner: Get years in the order they appear left-to-right
        years_in_order = [year for idx, year in sorted(col_map.items())]

        # 2. Data Extraction with Filtered Zip
        for row in rows:
            cells = row.find_all("td")
            if len(cells) < 2: continue
            
            metric = cells[0].get_text(strip=True)
            if not metric: continue

            numeric_values = []
            for cell in cells[1:]:
                val_text = cell.get_text(strip=True)
                # Filtering out (4), 11, $, etc. while keeping 162,560
                if re.search(r'\d{3,}', val_text):
                    numeric_values.append(val_text)
            
            
            for value, year in zip(numeric_values, years_in_order):
                sentence = f"The {metric} of {self.company} in the fiscal year {year} was {value}."
                results.append(sentence)
                
        return results

# EXTRACTION & CHUNKING

In [88]:
def extract_sections_from_xbrl_html(soup, table_converter) -> list[dict]:
    """Semantic section extraction with font-weight detection and table parsing."""
    sections, section_num = [], 0
    current_title, current_text, current_length = "Preamble", [], 0
    all_elements = soup.find_all(['span', 'div', 'p', 'ix:nonnumeric', 'ix:nonfraction', 'table'])
    
    for elem in all_elements:
        if elem.name == 'table':
            table_prose = table_converter.convert(str(elem))
            if not table_prose: continue
            text, is_header = " ".join(table_prose), False
        else:
            if elem.find(['span', 'div']): 
                continue
            text = elem.get_text(strip=True)
            if len(text) < 5: 
                continue
            style = elem.get('style', '')
            is_header = ('font-weight:700' in style and len(text) < 150 and 
                         any(text.upper().startswith(k) for k in ['ITEM', 'PART', 'CONSOLIDATED', 'NOTE']))
        
        if is_header and current_text and current_length > 200:
            section_num += 1
            sections.append({
                "section_num": section_num, 
                "section_title": current_title, 
                "text": " ".join(current_text), 
                "char_count": current_length
            })
            current_title, current_text, current_length = text[:100], [], 0
        else:
            current_text.append(text)
            current_length += len(text)
            if current_length > 1500:
                section_num += 1
                sections.append({
                    "section_num": section_num, 
                    "section_title": current_title, 
                    "text": " ".join(current_text), 
                    "char_count": current_length
                })
                current_text, current_length = [], 0
    
    if current_text:
        sections.append({
            "section_num": section_num + 1, 
            "section_title": current_title, 
            "text": " ".join(current_text), 
            "char_count": current_length
        })
    return sections

In [89]:
def build_chunks_with_citations(sections, company, doc_type, fiscal_year, source_file):
    """Quality filtering and metadata assignment."""
    financial_signals = ["revenue", "income", "profit", "sales", "earnings", "cash", "assets", "net", "total"]
    chunks = []
    for section in sections:
        if 'http://' in section["text"] or 'https://' in section["text"]: 
            continue
        quality_score = sum(1 for w in financial_signals if w in section["text"].lower()) / len(financial_signals)
        if quality_score < 0.2: 
            continue
        chunks.append({
            "chunk_id": f"{company.lower().replace(' ', '_')}_{fiscal_year}_s{section['section_num']:03d}",
            "text": section["text"], 
            "company": company, 
            "doc_type": doc_type, 
            "fiscal_year": fiscal_year,
            "section_title": section["section_title"], 
            "source_file": source_file, 
            "quality_score": round(quality_score, 3)
        })
    return chunks

# enrinching And  Storing in chroma db

In [90]:
def enrich_chunk_text(chunk):
    """Adds top-level context to every chunk for better embedding matching."""
    return f"{chunk['section_title']}:\n{chunk['text']}"

In [91]:
converter = TableToNaturalLanguage(company="Apple Inc.")
sections = extract_sections_from_xbrl_html(soup, converter)
chunks = build_chunks_with_citations(sections, "Apple Inc.", "10-K", "FY2023", "aapl-20230930.htm")
enriched_texts = [enrich_chunk_text(c) for c in chunks]

In [92]:
client = chromadb.PersistentClient(path="./chroma_db/")
try: 
    client.delete_collection("financial_docs_v2")
except: 
    pass
collection = client.create_collection(name="financial_docs_v2", metadata={"hnsw:space": "cosine"})

In [93]:
# Embedding
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(enriched_texts, batch_size=32, show_progress_bar=True)

Loading weights: 100%|█████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4367.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|███████████████████████████████████████████████████████████████████████| 3/3 [00:04<00:00,  1.47s/it]


In [94]:
collection.add(
    ids=[c["chunk_id"] for c in chunks],
    embeddings=embeddings.tolist(),
    documents=enriched_texts,
    metadatas=[{k: v for k, v in c.items() if k != 'text'} for c in chunks]
)

In [95]:
tokenized_enriched = [text.lower().split() for text in enriched_texts]
bm25 = BM25Okapi(tokenized_enriched)

# hybrid retrieval and prompting 

In [96]:
def expand_financial_query(query):
    expansions = {
        "revenue": "revenue net sales total sales",
        "profit": "profit net income earnings",
        "cash flow": "cash flow operations operating activities"
    }
    expanded = query
    for term, exp in expansions.items():
        if term in query.lower(): 
            expanded += f" {exp}"
    return expanded

In [97]:
def retreive_with_citations(query, bm25_index, enriched_docs, chunks, collection, n_results=3, alpha=0.7):
    tokenized_query = query.lower().split()
    bm25_scores = bm25_index.get_scores(tokenized_query)
    
    # Normalize BM25 scores
    bm_min, bm_max = bm25_scores.min(), bm25_scores.max()
    bm_norm = (bm25_scores - bm_min) / (bm_max - bm_min) if bm_max > bm_min else bm25_scores

    # Dense scores
    dense_results = collection.query(query_texts=[query], n_results=len(enriched_docs))
    dense_sims = 1 - np.array(dense_results["distances"][0])
    dense_ids = dense_results["ids"][0]
    
    dense_aligned = np.zeros(len(enriched_docs))
    for i, chunk in enumerate(chunks):
        if chunk["chunk_id"] in dense_ids:
            idx = dense_ids.index(chunk["chunk_id"])
            dense_aligned[i] = dense_sims[idx]

    hybrid_scores = (alpha * dense_aligned) + ((1 - alpha) * bm_norm)
    top_indices = np.argsort(-hybrid_scores)[:n_results]

    results = []
    for idx in top_indices:
        chunk = chunks[idx]
        results.append({
            "text": enriched_docs[idx],
            "citation": f"{chunk['company']} {chunk['doc_type']} {chunk['fiscal_year']} — {chunk['section_title']}",
            "hybrid_score": round(float(hybrid_scores[idx]), 4)
        })
    return results

In [99]:
def citation_formatted_answer(query, retrieved_results):
    context = "\n\n".join([f"[Source {i+1}: {r['citation']}]\n{r['text']}" for i, r in enumerate(retrieved_results)])
    return f"""You are a financial analyst. Answer using ONLY the sources below. Cite source numbers in brackets like [Source 1].

Sources:
{context}

Question: {query}
Answer:"""

In [100]:
query = "What is Apple's total revenue for 2023?"
expanded = expand_financial_query(query)
results = retreive_with_citations(expanded, bm25, enriched_texts, chunks, collection)
print(citation_formatted_answer(query, results))

You are a financial analyst. Answer using ONLY the sources below. Cite source numbers in brackets like [Source 1].

Sources:
[Source 1: Apple Inc. 10-K FY2023 — Note 2 –Revenue]
Note 2 –Revenue:
iPhone 200,583 200,583 205,489 205,489 191,973 191,973 29,357 29,357 40,177 40,177 35,190 35,190 28,300 28,300 29,292 29,292 31,862 31,862 Wearables, Home and Accessories 39,845 39,845 41,241 41,241 38,367 38,367 Services 85,200 85,200 78,129 78,129 68,425 68,425 Total net sales 383,285 383,285 394,328 394,328 365,817 365,817 Products net sales include amortization of the deferred value of unspecified software upgrade rights, which are bundled in the sales price of the respective product. Services net sales include amortization of the deferred value of services bundled in the sales price of certain products. Total net sales include $8.2billion of revenue recognized in 2023 that was included in deferred revenue as of September 24, 2022, $7.5billion of revenue recognized in 2022 that was included

In [101]:
query = "What is Apple's net income?"
expanded = expand_financial_query(query)
results = retreive_with_citations(expanded, bm25, enriched_texts, chunks, collection)
prompt = citation_formatted_answer(query, results)
print(prompt[:1500])

You are a financial analyst. Answer using ONLY the sources below. Cite source numbers in brackets like [Source 1].

Sources:
[Source 1: Apple Inc. 10-K FY2023 — Note 13 –Segment Information and Geographic Data]
Note 13 –Segment Information and Geographic Data:
Segment operating income 150,888 150,888 152,895 152,895 137,006 137,006 Research and development expense (29,915) 29,915 (26,251) 26,251 (21,914) 21,914 Other corporate expenses, net (6,672) 6,672 (7,207) 7,207 (6,143) 6,143 Total operating income 114,301 114,301 119,437 119,437 108,949 108,949 Includes corporate marketing expenses, certain share-based compensation expenses, various nonrecurring charges, and other separately managed general and administrative costs. Apple Inc. | 2023 Form 10-K | 47 The U.S. and China were the only countries that accounted for more than 10% of the Company’s net sales in 2023, 2022 and 2021.Net sales for 2023, 2022 and 2021 and long-lived assets as of September 30, 2023 and September 24, 2022 were

In [109]:
print(chunks[0].keys())
print(chunks[0])

dict_keys(['chunk_id', 'text', 'company', 'doc_type', 'fiscal_year', 'section_title', 'source_file', 'quality_score'])
{'chunk_id': 'apple_inc._FY2023_s012', 'text': 'AppleCare The Company offers a portfolio of fee-based service and support products under the AppleCare brand. The offerings provide priority access to Apple technical support, access to the global Apple authorized service network for repair and replacement services, and in many cases additional coverage for instances of accidental damage or theft and loss, depending on the country and type of product. Cloud Services The Company’s cloud services store and keep customers’ content up-to-date and available across multiple Apple devices and Windows personal computers. Digital Content The Company operates various platforms, including the App Store that allow customers to discover and download applications and digital content, such as books, music, video, games and podcasts. The Company also offers digital content through subscr

# Citation 

In [53]:
def build_chunks_with_citations(sections: list[dict],
                               company: str,
                               doc_type:str,
                               fiscal_year:str,
                               source_file:str) -> list[dict]:

    # every chunk must carry full provenance so LLM can say exactly where each answer came from
    financial_signals = [
        "revenue", "income", "profit", "loss", "sales", "earnings",
        "expense", "cost", "margin", "cash", "debt", "equity",
        "assets", "liabilities", "operating", "net", "gross", "total",
        "billion", "million", "fiscal", "quarter", "annual", "growth",
        "depreciation", "amortization", "ebitda", "eps", "dividend"
    ]

    chunks = []
    for section in sections:
        # WHY: XBRL metadata contains URLs, real financial text doesn't
        if 'http://' in section["text"] or 'https://' in section["text"]:
            continue
        text_lower = section["text"].lower()
        signals_found = sum(1 for w in financial_signals if w in text_lower)
        quality_score = signals_found/ len(financial_signals)

        if quality_score < 0.2:
            continue

        chunks.append({
            "chunk_id": f"{company.lower().replace(' ', '_').replace('.', '')}_{fiscal_year}_s{section['section_num']:03d}",            "text": section["text"],
            "char_count": section["char_count"],
            "company": company,
            "doc_type": doc_type,
            "fiscal_year": fiscal_year,
            "section_title": section["section_title"],
            "section_num": section["section_num"],
            "source_file": source_file,
            "quality_score": round(quality_score, 3)
        })

    return chunks

In [70]:
chunks = build_chunks_with_citations(
    sections,
    company="Apple Inc.",
    doc_type="10-K",
    fiscal_year="FY2023",
    source_file="aapl-20230930.htm"
)

In [73]:
print("Searching for revenue and net income chunks:")
for chunk in chunks:
    if "383,285" in chunk["text"] or "96,995" in chunk["text"]:
        print(f"\n  ID: {chunk['chunk_id']}")
        print(f"  Preview: {chunk['text'][:300]}")

Searching for revenue and net income chunks:

  ID: apple_inc_FY2023_s073
  Preview: (In millions, except number of shares, which are reflected in thousands, and per-share amounts) Years ended September 30,2023 September 24,2022 September 25,2021 Net sales: Products 298,085 298,085 316,199 316,199 297,392 297,392 Services 85,200 85,200 78,129 78,129 68,425 68,425 Total net sales 383

  ID: apple_inc_FY2023_s074
  Preview: (In millions) Years ended September 30,2023 September 24,2022 September 25,2021 Net income 96,995 96,995 99,803 99,803 94,680 94,680 Other comprehensive income/(loss): Change in foreign currency translation, net of tax (765) (1,511) 1,511 Change in unrealized gains/losses on derivative instruments, 

  ID: apple_inc_FY2023_s077
  Preview: (In millions, except per-share amounts) Years ended September 30,2023 September 24,2022 September 25,2021 Total shareholders’ equity, beginning balances 50,672 50,672 63,090 63,090 65,339 65,339 Common stock and additional paid-in ca

In [71]:
print(f"Total chunks after quality filter: {len(chunks)}")
print(f"\nSample citations:")
for chunk in chunks[:5]:
    print(f"\n  ID: {chunk['chunk_id']}")
    print(f"  Citation: {chunk['company']} {chunk['doc_type']} {chunk['fiscal_year']}")
    print(f"  Section: {chunk['section_title']}")
    print(f"  Quality: {chunk['quality_score']}")
    print(f"  Preview: {chunk['text'][:100]}")

Total chunks after quality filter: 43

Sample citations:

  ID: apple_inc_FY2023_s013
  Citation: Apple Inc. 10-K FY2023
  Section: PART I
  Quality: 0.241
  Preview: Markets and Distribution The Company’s customers are primarily in the consumer, small and mid-sized 

  ID: apple_inc_FY2023_s016
  Citation: Apple Inc. 10-K FY2023
  Section: PART I
  Quality: 0.241
  Preview: The Company regularly files patent, design, copyright and trademark applications to protect innovati

  ID: apple_inc_FY2023_s024
  Citation: Apple Inc. 10-K FY2023
  Section: PART I
  Quality: 0.207
  Preview: The Company’s ability to compete successfully depends heavily on ensuring the continuing and timely 

  ID: apple_inc_FY2023_s029
  Citation: Apple Inc. 10-K FY2023
  Section: PART I
  Quality: 0.207
  Preview: The Company’s products and services may be affected from time to time by design and manufacturing de

  ID: apple_inc_FY2023_s042
  Citation: Apple Inc. 10-K FY2023
  Section: PART I
  Quality: 0.276


In [56]:
# Find the key financial statement chunks
print("Financial statement chunks:")
key_sections = [
    "CONSOLIDATED STATEMENTS OF OPERATIONS",
    "CONSOLIDATED BALANCE SHEETS", 
    "CONSOLIDATED STATEMENTS OF CASH FLOWS",
    "Note 2",
    "Note 13"
]

for chunk in chunks:
    if any(key in chunk['section_title'] for key in key_sections):
        print(f"\n  ID: {chunk['chunk_id']}")
        print(f"  Section: {chunk['section_title']}")
        print(f"  Quality: {chunk['quality_score']}")
        print(f"  Preview: {chunk['text'][:150]}")

Financial statement chunks:

  ID: apple_inc_FY2023_s073
  Section: CONSOLIDATED STATEMENTS OF OPERATIONS
  Quality: 0.379
  Preview: (In millions, except number of shares, which are reflected in thousands, and per-share amounts) Years ended September 30,2023 September 24,2022 Septem

  ID: apple_inc_FY2023_s075
  Section: CONSOLIDATED BALANCE SHEETS
  Quality: 0.31
  Preview: (In millions, except number of shares, which are reflected in thousands, and par value) September 30,2023 September 24,2022 ASSETS: Current assets: Ca

  ID: apple_inc_FY2023_s078
  Section: CONSOLIDATED STATEMENTS OF CASH FLOWS
  Quality: 0.414
  Preview: (In millions) Years ended September 30,2023 September 24,2022 September 25,2021 Cash, cash equivalents and restricted cash, beginning balances 24,977 

  ID: apple_inc_FY2023_s084
  Section: Note 2 –Revenue
  Quality: 0.241
  Preview: For the sale of third-party products where the Company obtains control of the product before transferring it to the customer, th

# Store in the chroma DB

In [74]:
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading embedding model...


Loading weights: 100%|█████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3424.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [75]:
texts = [chunk["text"] for chunk in chunks ]

In [76]:
print(f"Embedding {len(chunks)} chunks...")

Embedding 43 chunks...


# BM25 index over new chunks

In [77]:
tokenized_docs = [text.lower().split() for text in texts]
bm25 = BM25Okapi(tokenized_docs)
print(f"BM25 index build : {len(bm25.idf)} unique terms")

BM25 index build : 2480 unique terms


In [78]:
def retreive_with_citations(query: str, 
                             bm25_index, 
                             documents: list,
                             chunks: list,
                             collection,
                             n_results: int = 3,
                             alpha: float = 0.7) -> list[dict]:
    """
    We need citation because it will be easy for the use to verify whether the answer is genuine or not 
    """

    tokenized_query = query.lower().split()
    bm25_scores = bm25_index.get_scores(tokenized_query)
    bm25_min, bm25_max = bm25_scores.min(), bm25_scores.max()
    if bm25_max > bm25_min:
        bm25_normalized = (bm25_scores - bm25_min) / (bm25_max - bm25_min)
    else:
        bm25_normalized = bm25_scores

    # Dense scores
    dense_results = collection.query(
        query_texts=[query],
        n_results=len(documents)
    )
    dense_distances = np.array(dense_results["distances"][0])
    dense_similarity = 1 - dense_distances
    dense_ids = dense_results["ids"][0]
    all_ids = [chunk["chunk_id"] for chunk in chunks]

    dense_aligned = np.zeros(len(documents))
    for i, doc_id in enumerate(all_ids):
        if doc_id in dense_ids:
            dense_idx = dense_ids.index(doc_id)
            dense_aligned[i] = dense_similarity[dense_idx]

    # Hybrid scores
    hybrid_scores = (alpha * dense_aligned) + ((1 - alpha) * bm25_normalized)
    top_indices = np.argsort(-hybrid_scores)[:n_results]

    results = []
    for idx in top_indices:
        chunk = chunks[idx]
        
        # Build citation string
        # WHY formatted citation: LLM pastes this directly into answer
        citation = (f"{chunk['company']} {chunk['doc_type']} "
                   f"{chunk['fiscal_year']} — {chunk['section_title']}")
        
        results.append({
            "text": chunk["text"],
            "citation": citation,
            "company": chunk["company"],
            "doc_type": chunk["doc_type"],
            "fiscal_year": chunk["fiscal_year"],
            "section_title": chunk["section_title"],
            "hybrid_score": round(float(hybrid_scores[idx]), 4),
            "chunk_id": chunk["chunk_id"]
        })
    
    return results


In [79]:
test_queries = [
    "What is Apple's total revenue for 2023?",
    "What is Apple's net income?",
    "Apple cash flow from operations"]

In [69]:
for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = retreive_with_citations(query, bm25, texts, chunks, collection)
    for r in results[:2]:
        print(f"  [{r['citation']}]")
        print(f"  Score: {r['hybrid_score']} | Preview: {r['text'][:100]}")


Query: 'What is Apple's total revenue for 2023?'
  [Apple Inc. 10-K FY2023 — Note 2 –Revenue]
  Score: 0.7237 | Preview: iPhone 200,583 200,583 205,489 205,489 191,973 191,973 29,357 29,357 40,177 40,177 35,190 35,190 28,
  [Apple Inc. 10-K FY2023 — Item 6.    [Reserved]]
  Score: 0.6555 | Preview: Apple Inc. | 2023 Form 10-K | 19 Item 7.    Management’s Discussion and Analysis of Financial Condit

Query: 'What is Apple's net income?'
  [Apple Inc. 10-K FY2023 — Note 13 –Segment Information and Geographic Data]
  Score: 0.685 | Preview: Segment operating income 150,888 150,888 152,895 152,895 137,006 137,006 Research and development ex
  [Apple Inc. 10-K FY2023 — PART I]
  Score: 0.6432 | Preview: The Company regularly files patent, design, copyright and trademark applications to protect innovati

Query: 'Apple cash flow from operations'
  [Apple Inc. 10-K FY2023 — Note 8 –Leases]
  Score: 0.5313 | Preview: Note 9 –Debt Commercial Paper The Company issues unsecured short-term promisso

In [ ]:
def enrich_chunk_text(chunk: dict) -> str:
    """
    WHY prepend section title: tables have no natural language
    adding the title gives the embedding model context about
    what the numbers mean
    """
    return f"{chunk['section_title']}:\n{chunk['text']}"

# Re-embed with enriched text
enriched_texts = [enrich_chunk_text(chunk) for chunk in chunks]

# Rebuild collection with enriched embeddings
try:
    client.delete_collection("financial_docs_v2")
except:
    pass

collection = client.create_collection(
    name="financial_docs_v2",
    metadata={"hnsw:space": "cosine"}
)

print("Re-embedding with enriched text...")
enriched_embeddings = model.encode(enriched_texts, batch_size=32, show_progress_bar=True)

collection.add(
    ids=[chunk["chunk_id"] for chunk in chunks],
    embeddings=enriched_embeddings.tolist(),
    documents=enriched_texts,
    metadatas=[{
        "company": chunk["company"],
        "doc_type": chunk["doc_type"],
        "fiscal_year": chunk["fiscal_year"],
        "section_title": chunk["section_title"],
        "section_num": chunk["section_num"],
        "source_file": chunk["source_file"],
        "quality_score": chunk["quality_score"]
    } for chunk in chunks]
)

# Rebuild BM25 with enriched text
tokenized_enriched = [text.lower().split() for text in enriched_texts]
bm25 = BM25Okapi(tokenized_enriched)

print(f"Re-indexed {collection.count()} chunks with enriched text")

In [ ]:
for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = retreive_with_citations(query, bm25, enriched_texts, chunks, collection)
    for r in results[:2]:
        print(f"  [{r['citation']}]")
        print(f"  Score: {r['hybrid_score']} | Preview: {r['text'][:120]}")

In [ ]:
# Direct lookup — are the financial statement chunks even in the collection?
print("Direct retrieval test — searching for income statement chunk:")
result = collection.get(
    ids=["apple_inc_FY2023_s071"],
    include=["documents", "metadatas"]
)
print(f"Found: {len(result['ids'])} chunk(s)")
if result['ids']:
    print(f"Section: {result['metadatas'][0]['section_title']}")
    print(f"Preview: {result['documents'][0][:200]}")

print("\nDirect query with exact financial terms:")
direct = collection.query(
    query_texts=["total net sales 383285 revenue products services"],
    n_results=3
)
for i in range(len(direct['ids'][0])):
    meta = direct['metadatas'][0][i]
    print(f"  Section: {meta['section_title']} | Distance: {round(direct['distances'][0][i], 4)}")

In [ ]:
def expand_financial_query(query: str) -> str:
    """
    query expansion: bridges vocabulary gap between
    natural language questions and numeric financial tables
    """
    expansions = {
        "revenue": "revenue net sales total sales income",
        "profit": "profit net income earnings",
        "cash flow": "cash flow operations operating activities",
        "expenses": "expenses cost of sales operating expenses",
        "assets": "assets current assets total assets",
        "debt": "debt liabilities long-term debt",
        "earnings per share": "earnings per share EPS diluted basic",
        "net income": "net income profit earnings after tax",
        "total revenue": "total revenue net sales total net sales",
    }
    
    expanded = query
    query_lower = query.lower()
    
    for term, expansion in expansions.items():
        if term in query_lower:
            expanded = f"{expanded} {expansion}"
    
    return expanded

# Test query expansion
test_queries_expanded = [
    "What is Apple's total revenue for 2023?",
    "What is Apple's net income?",
    "Apple cash flow from operations"
]

print("Query expansion results:")
for q in test_queries_expanded:
    expanded = expand_financial_query(q)
    print(f"\nOriginal: {q}")
    print(f"Expanded: {expanded}")

In [ ]:
# Test retrieval with expanded queries
print("Retrieval with query expansion:")
for query in test_queries_expanded:
    expanded = expand_financial_query(query)
    results = retreive_with_citations(expanded, bm25, enriched_texts, chunks, collection)
    print(f"\nQuery: '{query}'")
    for r in results[:2]:
        print(f"  [{r['citation']}]")
        print(f"  Score: {r['hybrid_score']} | Preview: {r['text'][:100]}")

# Citation Formatter 

In [ ]:
def citation_formatted_answer(query: str,
                             retrieved_results: list[dict]) -> str:
    context_parts = []
    
    for i, result in enumerate(retrieved_results):
        context_parts.append(
            f"[Source {i+1}: {result['citation']}]\n{result['text'][:500]}"
        )
    
    context = "\n\n".join(context_parts)
    
    prompt = f"""You are a financial analyst assistant. Answer the question using ONLY the provided sources below.
For every fact you state, cite the source number in brackets like [Source 1].
If the sources don't contain enough information, say so explicitly.

Sources:
{context}

Question: {query}

Answer:"""
    
    return prompt

In [ ]:
query = "What is Apple's cash flow from operations?"
expanded_query = expand_financial_query(query)
results = retreive_with_citations(expanded_query, bm25, enriched_texts, chunks, collection)
prompt = citation_formatted_answer(query, results)
print("Citation-aware prompt:")
print("-" * 50)
print(prompt[:800])
print("...")
print(f"\nTotal prompt length: {len(prompt)} chars")

## Day 63  Better Chunking + Citation Layer

### What I built
- Semantic section extraction using font-weight:700 detection
- XBRL metadata filter (URL detection)
- Citation metadata: company, doc_type, fiscal_year, section_title
- Section title enrichment before embedding
- Query expansion for financial vocabulary
- Citation-aware prompt formatter

### Real problems discovered
1. XBRL format: Apple 10-K uses inline XBRL — no standard HTML tags
2. Tables don't embed well: numbers without prose context score poorly
3. Query-document mismatch: "revenue" query misses income statement table
4. Partial fixes: URL filter, title enrichment, query expansion

### What works
- Cash flow query → CONSOLIDATED STATEMENTS OF CASH FLOWS ✅
- Citation format: "Apple Inc. 10-K FY2023 — Section Title" ✅
- Prompt structure ready for LLM generation ✅

### Root cause of retrieval failures
Financial tables contain numbers, not natural language.
Embedding models trained on text don't represent numbers well.
Fix: convert tables to natural language summaries before embedding.
This is Day 64's work.

### Production insight
Query expansion: expand for retrieval, use original for generation.
Retrieval and generation have different vocabulary needs.